# Movies and Audience Ratings: Initial Data Profile and Cleaning Plan
**Daniel Yarritu | BAN 6003 | October 2, 2026**

This submission completes Milestone 1. The five supplied tables contain 9,082 movies and 99,810 rating events. Initial checks found no duplicate records or expected keys, but missing descriptive fields, zero financial/runtime values, and uneven rating coverage require preparation. The planned final analytic base table (ABT) will contain one row per movie. Later milestone sections remain starter material and are not completed results.


## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)

# Later milestone examples are preserved but not executed in this submission.
RUN_FUTURE_MILESTONES = False


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3).iloc[:, :6])  # bounded preview; full schema appears below


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language
0,1,862,tt0114709,Toy Story,Toy Story,en
1,2,8844,tt0113497,Jumanji,Jumanji,en
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en


ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02


links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0


credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret


keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Business problem, question, and decision context
A streaming service catalog analyst needs to understand audience reception when reviewing which types of existing titles to investigate for its catalog. **Which movie characteristics are associated with higher average audience ratings, and how well can genre, runtime, release year, and original language predict average MovieLens ratings for movies with at least 20 ratings?** This is a retrospective study of the provided catalog, not proof that a characteristic causes higher ratings or a complete acquisition decision. Licensing costs, current demand, and audience fit are outside this dataset.

### Dataset and sources
Use the professor's [Option 1 template](https://github.com/tianhaiz/ban6003-project-option-1-movies-template) and its [data guide](https://github.com/tianhaiz/ban6003-project-option-1-movies-template/blob/main/data/README.md). The starter describes these curated files as The Movies Dataset / MovieLens ratings. The supplied subset, rather than an independently downloaded full dataset, controls this project. Exact raw-file SHA-256 hashes are saved below to identify the inputs.

| Raw file | What one row means | Expected key | Planned use |
|---|---|---|---|
| movies_metadata.csv | One movie in the curated catalog | movieId; also unique tmdbId | Base movie table; genre, runtime, year, language |
| ratings.csv | One user's rating event for one movie | userId + movieId | Aggregate ratings before a movie-level join |
| links.csv | One MovieLens-to-IMDb/TMDB crosswalk | movieId | Validate identifier mappings and coverage |
| credits_curated.csv | One movie's director and top cast | tmdbId | Optional descriptive enrichment |
| keywords_curated.csv | One movie's keyword text | tmdbId | Optional descriptive enrichment |

The profile and key audit above give initial counts and duplicates. The following checks cover every column's type and missingness, plausible value ranges, date parsing, text consistency, empty list-like fields, and cross-table coverage. Raw files remain unchanged.

### Proposed outcome and preparation assumptions
The primary candidate outcome is the **mean MovieLens rating per movie**, measured on a 0.5-5 scale. Calculate it from valid rating events only after checking duplicate user-movie pairs. Initially require at least 20 ratings for modeling, following the starter's evidence threshold. Examine alternative thresholds later because this rule favors better-covered films. Rating count is an eligibility/uncertainty measure, not a predictor. TMDB vote_average uses a different 0-10 scale and must not be substituted for this outcome.

The final ABT is planned as one row per movie keyed by movieId. Summarize ratings by movieId before joining; join credits and keywords on unique tmdbId. Check row counts, key uniqueness, and unmatched rows at each integration step. Movie titles alone are not keys because different films can share titles.

### Initial cleaning plan
1. Preserve raw CSVs. Create separate cleaned copies and record each transformation and affected row count.
2. Validate identifiers before converting links.tmdbId from float to nullable integer; keep missing IDs missing. Preserve leading-zero IMDb identifiers as strings when constructing a crosswalk.
3. Parse release_date with coercion and compare its year to release_year. Keep raw timestamps; verify rating_datetime against Unix seconds in UTC. Treat parsing failures as issues to investigate.
4. Treat zero budget/revenue as unknown-like values pending source clarification rather than evidence of no cost or no earnings. These financial fields are not needed for the initial rating model. Convert runtime=0 to missing in the cleaned copy; inspect very long runtimes before excluding anything because documentaries/series may be valid.
5. Trim text and inspect categories before recoding. Missing genre, director, cast, status, and keywords remain unknown, not invented facts. Pipe-delimited genres/cast/keywords need deliberate splitting; parse Python-like nested strings with ast.literal_eval, never eval. Preserve multi-genre membership rather than assuming primary_genre captures every genre.
6. Keep legitimate movie records with incomplete descriptive fields during profiling. Later, choose and document an unknown category or train-only imputation for selected model inputs. If duplicates arise, inspect them before selecting a record; do not automatically drop repeated titles.
7. Aggregate valid ratings, apply the evidence threshold, then validate the movie-level ABT and document all retained columns. Exclude rating summaries, rating dates, TMDB voting measures, popularity, and observed revenue from the initial features to avoid outcome leakage or circular predictions.

### Ethics, privacy, governance, and limitations
User IDs are pseudonymous, not proof of anonymity. Do not attempt re-identification or combine them with personal data. Report movie-level summaries rather than individual user behavior. These historical ratings reflect self-selected users and uneven coverage, not all viewers or present-day streaming demand. A 20-rating threshold improves evidence volume but does not guarantee reliability and introduces selection bias. Missing metadata may differ by language or era; examine coverage by those groups before broad recommendations. Ratings measure reported preference, not objective artistic quality. Confirm upstream reuse/licensing conditions before redistribution beyond this class; the template does not establish blanket commercial permission.


In [6]:
# Full column-level profile (before cleaning)
column_profile = pd.concat([
    pd.DataFrame({
        'table': name, 'column': df.columns,
        'dtype': [str(t) for t in df.dtypes],
        'missing_count': df.isna().sum().values,
        'missing_percent': (100 * df.isna().mean()).round(2).values,
        'distinct_nonmissing': df.nunique().values,
    }) for name, df in tables.items()
], ignore_index=True)
display(column_profile)
column_profile.to_csv(OUTPUT_DIR / 'milestone1_column_profile.csv', index=False)


,table,column,dtype,missing_count,missing_percent,distinct_nonmissing
0,movies,movieId,int64,0,0.00,9082
1,movies,tmdbId,int64,0,0.00,9082
2,movies,imdb_id,str,0,0.00,9082
3,movies,title,str,0,0.00,8809
4,movies,original_title,str,0,0.00,8841
5,movies,original_language,str,0,0.00,42
6,movies,release_date,str,0,0.00,6002
7,movies,runtime,float64,0,0.00,235
8,movies,budget,int64,0,0.00,599
9,movies,revenue,float64,0,0.00,4555


In [7]:
# Suspicious values and timestamp/date consistency
release_dates = pd.to_datetime(movies['release_date'], errors='coerce')
rating_dates = pd.to_datetime(ratings['rating_datetime'], errors='coerce', utc=True)
unix_dates = pd.to_datetime(ratings['timestamp'], unit='s', errors='coerce', utc=True)
checks = {
    'budget equals zero (review unknown-like values)': movies['budget'].eq(0).sum(),
    'revenue equals zero (review unknown-like values)': movies['revenue'].eq(0).sum(),
    'runtime equals zero': movies['runtime'].eq(0).sum(),
    'runtime exceeds 300 minutes (review, not automatic error)': movies['runtime'].gt(300).sum(),
    'negative budget, revenue, or runtime': movies[['budget','revenue','runtime']].lt(0).any(axis=1).sum(),
    'invalid release dates': release_dates.isna().sum(),
    'release year differs from parsed date': (release_dates.notna() & release_dates.dt.year.ne(movies['release_year'])).sum(),
    'ratings outside 0.5-5 or half-point increments': (~ratings['rating'].between(0.5,5) | ratings['rating'].mul(2).mod(1).ne(0)).sum(),
    'invalid rating dates or timestamps': (rating_dates.isna() | unix_dates.isna()).sum(),
    'rating datetime differs from Unix timestamp': (rating_dates.notna() & unix_dates.notna() & rating_dates.ne(unix_dates)).sum(),
    'TMDB vote_average outside 0-10': (~movies['vote_average'].between(0,10)).sum(),
}
suspicious_checks = pd.Series(checks, name='affected_rows').to_frame()
display(suspicious_checks)
display(movies.loc[movies['runtime'].gt(300), ['movieId','title','runtime']])
suspicious_checks.to_csv(OUTPUT_DIR / 'milestone1_value_checks.csv')


,affected_rows
budget equals zero (review unknown-like values),4443
revenue equals zero (review unknown-like values),4322
runtime equals zero,20
"runtime exceeds 300 minutes (review, not automatic error)",15
"negative budget, revenue, or runtime",0
invalid release dates,0
release year differs from parsed date,0
ratings outside 0.5-5 or half-point increments,0
invalid rating dates or timestamps,0
rating datetime differs from Unix timestamp,0


,movieId,title,runtime
4862,6896,Shoah,566.0
5213,7762,Tinker Tailor Soldier Spy,320.0
5216,7767,The Best of Youth,366.0
5870,27002,From the Earth to the Moon,720.0
5881,27251,The 10th Kingdom,417.0
6234,38499,Angels in America,352.0
6899,58649,Napoleon,332.0
7224,69241,Berlin Alexanderplatz,931.0
7767,86487,Mildred Pierce,336.0
7981,93040,The Civil War,680.0


In [8]:
# Coverage and text/category checks; this is profiling, not ABT construction.
coverage = pd.Series({
    'rating events with no matching movie': (~ratings['movieId'].isin(movies['movieId'])).sum(),
    'catalog movies without ratings': (~movies['movieId'].isin(ratings['movieId'])).sum(),
    'link rows outside curated movie catalog': (~links['movieId'].isin(movies['movieId'])).sum(),
    'movies without credits match': (~movies['tmdbId'].isin(credits['tmdbId'])).sum(),
    'movies without keywords match': (~movies['tmdbId'].isin(keywords['tmdbId'])).sum(),
}, name='affected_rows')
display(coverage.to_frame())
mapping_check = movies[['movieId','tmdbId']].merge(
    links[['movieId','tmdbId']], on='movieId', how='left',
    suffixes=('_movies','_links'), validate='one_to_one')
display(pd.Series({
    'movie rows missing linked TMDB ID': mapping_check['tmdbId_links'].isna().sum(),
    'conflicting nonmissing TMDB mappings': (mapping_check['tmdbId_links'].notna() & mapping_check['tmdbId_movies'].ne(mapping_check['tmdbId_links'])).sum(),
}).to_frame('affected_rows'))
text_checks = []
for name, df in tables.items():
    for column in df.select_dtypes(include=['object','string']).columns:
        values = df[column].dropna().astype(str)
        text_checks.append({'table':name, 'column':column,
            'outer_whitespace': int(values.ne(values.str.strip()).sum()),
            'blank_strings': int(values.str.strip().eq('').sum()),
            'empty_list_strings': int(values.str.strip().eq('[]').sum())})
display(pd.DataFrame(text_checks))
display(movies['status'].value_counts(dropna=False).to_frame('movie_count'))
display(movies['primary_genre'].value_counts(dropna=False).to_frame('movie_count'))
rating_counts = ratings.groupby('movieId').size()
display(pd.Series({
    'movies with any ratings': len(rating_counts),
    'movies with at least 20 ratings': rating_counts.ge(20).sum(),
    'rated movies below threshold': rating_counts.lt(20).sum(),
    'median ratings per rated movie': rating_counts.median(),
    'minimum rating': ratings['rating'].min(),
    'maximum rating': ratings['rating'].max(),
    'earliest rating (UTC)': rating_dates.min(),
    'latest rating (UTC)': rating_dates.max(),
}).to_frame('value'))
# Record raw-file versions for reproducibility.
import hashlib, platform
raw_manifest = pd.DataFrame([
    {'file': str(path.relative_to(REPO_ROOT)),
     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    for path in sorted(DATA_DIR.glob('*.csv'))
])
raw_manifest.to_csv(OUTPUT_DIR / 'milestone1_raw_file_manifest.csv', index=False)
key_audit.to_csv(OUTPUT_DIR / 'milestone1_key_audit.csv', index=False)
print(f'Python {platform.python_version()}; pandas {pd.__version__}; numpy {np.__version__}')


,affected_rows
rating events with no matching movie,0
catalog movies without ratings,57
link rows outside curated movie catalog,43
movies without credits match,0
movies without keywords match,0


,affected_rows
movie rows missing linked TMDB ID,0
conflicting nonmissing TMDB mappings,0


,table,column,outer_whitespace,blank_strings,empty_list_strings
0,movies,imdb_id,0,0,0
1,movies,title,0,0,0
2,movies,original_title,0,0,0
3,movies,original_language,0,0,0
4,movies,release_date,0,0,0
5,movies,genres,0,0,35
6,movies,production_companies,0,0,664
7,movies,production_countries,0,0,294
8,movies,status,0,0,0
9,movies,genres_list,0,0,0


,movie_count
status,
Released,9061
Rumored,11
Post Production,7
NaN,2
In Production,1


,movie_count
primary_genre,
Drama,2413
Comedy,2168
Action,1177
Adventure,517
Horror,497
Crime,417
Documentary,383
Thriller,263
Fantasy,244


,value
movies with any ratings,9025
movies with at least 20 ratings,1300
rated movies below threshold,7725
median ratings per rated movie,3.0
minimum rating,0.5
maximum rating,5.0
earliest rating (UTC),1995-01-09 11:46:49+00:00
latest rating (UTC),2016-10-16 17:57:24+00:00


Python 3.12.13; pandas 3.0.3; numpy 2.4.6


### What the initial checks mean
The raw tables have the following dimensions: movies **9,082 x 21**, ratings **99,810 x 5**, links **9,125 x 3**, credits **9,082 x 3**, and keywords **9,082 x 2**. No full-row duplicates or duplicate expected keys were found. Repeated movieId values across ratings are expected because multiple users rate the same movie.

Movie metadata has 2 missing status values and 35 missing values each in genres_list and primary_genre. Credits have 24 missing directors and 90 missing cast fields; keywords are missing for 761 movies. Links has 13 missing tmdbId values, which makes pandas infer float64; these are identifiers, not measurements. release_date is initially object/string and requires explicit parsing. The nested metadata columns and pipe-delimited lists are text rather than ready-to-model features. Empty list strings can also represent absent metadata even when pandas does not mark a cell missing.

There are **4,443 zero budgets**, **4,322 zero revenues**, and **20 zero runtimes**. Zero financial values need source clarification; zero runtime is unsuitable as an observed film length. Fifteen runtimes exceed 300 minutes, including documentaries and series-like titles. They are review flags rather than proven errors. All supplied ratings fall between 0.5 and 5 in half-point increments. Release dates parse successfully and their years agree with release_year.

All rating events match the movie catalog. **57 catalog movies have no ratings**, and **43 link rows are outside this curated movie table**. Extra crosswalk rows are a coverage difference, not duplicate movies. Only **1,300 movies have at least 20 ratings**; the median rated movie has only **3 ratings**. This limits the initial model population and means a high average from one or two people should not be treated as dependable evidence. Coverage checks and the 20-rating rule will be revisited during integration.

### Next milestone
Implement the documented cleaning decisions on copies, aggregate valid ratings to movieId, and build/validate a one-row-per-movie ABT. This submission does not claim an estimated genre effect, model performance, or final catalog recommendation.


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [9]:
if RUN_FUTURE_MILESTONES:
    # Milestone 2 code workspace
    # Build or revise your ABT here.
    # Add row-count checks and duplicate-key checks after each major merge.
    
    pass

In [10]:
if RUN_FUTURE_MILESTONES:
    rating_summary = (
        ratings.groupby("movieId")
        .agg(
            user_rating_mean=("rating", "mean"),
            user_rating_count=("rating", "size"),
            user_rating_std=("rating", "std"),
            first_rating_date=("rating_datetime", "min"),
            last_rating_date=("rating_datetime", "max")
        )
        .reset_index()
    )
    
    abt = (
        movies
        .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
        .merge(credits, on="tmdbId", how="left", validate="one_to_one")
        .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
    )
    
    abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
    abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
    abt["high_user_rating"] = (
        abt["user_rating_mean"].ge(3.75)
        .where(abt["rating_target_eligible"])
        .astype("Int64")
    )
    pass

In [11]:
if RUN_FUTURE_MILESTONES:
    abt_validation = pd.Series({
        "movie_rows_before_merges": len(movies),
        "abt_rows_after_merges": len(abt),
        "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
        "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
        "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
        "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
    })
    
    display(abt_validation)
    display(abt.head())
    display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))
    pass

### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [12]:
if RUN_FUTURE_MILESTONES:
    import sqlite3
    
    with sqlite3.connect(":memory:") as connection:
        abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
        sql_abt_check = pd.read_sql_query(
            """
            SELECT
                COUNT(*) AS abt_rows,
                COUNT(DISTINCT movieId) AS unique_movie_ids,
                SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
            FROM movies_abt;
            """,
            connection,
        )
    
    sql_abt_check
    pass

### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [13]:
if RUN_FUTURE_MILESTONES:
    data_dictionary = pd.DataFrame({
        "column_name": abt.columns,
        "data_type": [str(abt[column].dtype) for column in abt.columns],
        "description": "",
        "source": "",
        "transformation_or_derivation": "",
        "notes_or_concerns": "",
    })
    
    # Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
    data_dictionary.head()
    pass

## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [14]:
if RUN_FUTURE_MILESTONES:
    # Milestone 3 code workspace
    # Suggested sequence:
    # 1. State the prediction or analysis timing in markdown.
    # 2. Filter to rows where your target is defined.
    # 3. Select only features available at that time.
    # 4. Check class balance or the target distribution.
    # 5. Create a reproducible train/test split when modeling.
    # 6. Fit a simple baseline and evaluate it on test data.
    pass

In [15]:
if RUN_FUTURE_MILESTONES:
    # Example placeholder. Replace with your chosen target and feature set.
    # target = 'your_target_column'
    # features = ['feature_1', 'feature_2']
    # model_df = abt[[target] + features].dropna()
    
    abt.describe(include='all').T.head(25)
    pass

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [16]:
if RUN_FUTURE_MILESTONES:
    # Final project code workspace
    # Save final datasets, figures, or tables here when they are ready.
    # Recommended filenames:
    # final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
    # dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
    # metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
    # report_path = REPORTS_DIR / "movies_final_report.html"
    pass

In [17]:
if RUN_FUTURE_MILESTONES:
    # Save final deliverables only after your ABT and documentation are complete.
    # abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
    # data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)
    
    # Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
    pass